In [1]:
import cv2
import pickle
import numpy as np

video_path = 'parking_3.mp4'

# trying to get existing markup with coordinates of parking slots
try:
    with open('slots.pkl', 'rb') as f:
        slots_list = pickle.load(f)
except:
    slots_list = []

current_points = []

def mouse_click(events, x, y, flags, param):
    global current_points

    if events == cv2.EVENT_LBUTTONDOWN:
        current_points.append((x, y))
        # if we set 4 points - save like one slot
        if len(current_points) == 4:
            slots_list.append(current_points)
            current_points = []

    if events == cv2.EVENT_RBUTTONDOWN:
        # remove last one slot
        if slots_list:
            slots_list.pop()

    with open('slots.pkl', 'wb') as f:
        pickle.dump(slots_list, f)

cap = cv2.VideoCapture(video_path)
success, frame = cap.read() # get only first frame

while True:
    img_display = frame.copy()

    # plot slots
    for slot in slots_list:
        pts = np.array(slot, np.int32)
        cv2.polylines(img_display, [pts], isClosed=True, color=(255, 0, 255), thickness=2)

    # plot dots
    for pt in current_points:
        cv2.circle(img_display, pt, 5, (0, 255, 0), -1)

    cv2.imshow("Image", img_display)
    cv2.setMouseCallback("Image", mouse_click)

    # q - quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [2]:
from ultralytics import YOLO
import pickle
import cv2
import numpy as np

# load the model
model = YOLO('yolo11s.pt')

# load coordinates of parking slots
with open('slots.pkl', 'rb') as f:
        slots_list = pickle.load(f)

video_path = 'parking_3.mp4'

cap = cv2.VideoCapture(video_path)

# read video
while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    # detection of different classes (car, truck etc)
    results = model(frame, device='cpu', conf=0.15, classes=[2, 5, 7, 67], verbose=False)[0]
    detections = results.boxes.xyxy.cpu().numpy()

    car_centers = []
    for det in detections:
        x1, y1, x2, y2 = det
        cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)
        car_centers.append((cx, cy))

        # plot this center
        cv2.circle(frame, (cx, cy), 4, (255, 0, 0), -1)

    for slot in slots_list:
        is_occupied = False

        # convert coordinates to numpy
        polygon = np.array(slot, np.int32)

        for center in car_centers:
            dist = cv2.pointPolygonTest(polygon, center, False) # return +1 if point inside slot
            if dist >= 0:
                is_occupied = True
                break

        color = (0, 0, 255) if is_occupied else (0, 255, 0)
        thickness = 2 if not is_occupied else 4

        # plot slot
        cv2.polylines(frame, [polygon], isClosed=True, color=color, thickness=thickness)

        # add labels
        label = "Occupied" if is_occupied else "Free"
        cv2.putText(frame, label, (slot[0][0], slot[0][1] - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    cv2.imshow("Parking Detector", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()